<a href="https://colab.research.google.com/github/gurveervirk/CodeSage/blob/main/GRPO_text2cypher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [1]:
%%capture
# Skip restarting message in Colab
import sys; modules = list(sys.modules.keys())
for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
!pip install unsloth vllm==0.7.2 neo4j deepdiff llama_index
!pip install --upgrade pillow

### Unsloth

In [1]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
max_seq_length = 4096 + 512 # Can increase for longer reasoning traces
lora_rank = 64 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Gurveer05/Qwen2.5-Coder-3B-Instruct-Text2Cypher",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.75, # Reduce if out of memory
)

# model = FastLanguageModel.get_peft_model(
#    model,
#    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
#    target_modules = [
#        "q_proj", "k_proj", "v_proj", "o_proj",
#        "gate_proj", "up_proj", "down_proj",
#    ], # Remove QKVO if out of memory
#    lora_alpha = lora_rank,
#    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
#    random_state = 3407,
#)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 03-19 14:24:20 __init__.py:190] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.16: Fast Qwen2 patching. Transformers: 4.48.3. vLLM: 0.7.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-coder-3b-instruct-bnb-4bit with actual GPU utilization = 74.29%
Unsloth: Your GPU has CUDA compute capability 7.5 with VRAM = 14.74 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 4608. Num Sequences = 224.
Unsloth: vLLM's KV Cache can use up to 8.53 GB. Also

tokenizer_config.json:   0%|          | 0.00/7.51k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

... (13 outputs left out)

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/479M [00:00<?, ?B/s]

Unsloth 2025.3.16 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


### Neo4j Code

In [2]:
#@title Neo4j Connection Code

from typing import Any, Dict, Optional, List
import neo4j

class Neo4jConnection:
    def __init__(self, uri, user, password, database):
        self._driver = neo4j.GraphDatabase.driver(uri, auth=(user, password))
        self._database = database
        self._timeout = None

    def close(self):
        self._driver.close()

    def structured_query(
        self,
        query: str
    ) -> Dict[str, Any]:
        try:
            # Try executing the query
            data, _, _ = self._driver.execute_query(
                neo4j.Query(text=query, timeout=self._timeout),
                database_=self._database
            )
            full_result = [d.data() for d in data]

            return {
                "success": True,
                "data": full_result,
                "error": None
            }

        except neo4j.exceptions.Neo4jError as e:
            # Capture and return the error details in JSON format
            error_response = {
                "success": False,
                "data": [],
                "error": {
                    "code": e.code,
                    "message": e.message
                }
            }

        # Fallback to allow implicit transactions
        try:
            with self._driver.session(database=self._database) as session:
                data = session.run(neo4j.Query(text=query, timeout=self._timeout))
                full_result = [d.data() for d in data]
                return {
                    "success": True,
                    "data": full_result,
                    "error": None
                }

        except neo4j.exceptions.Neo4jError as e:
            # If the fallback also fails, return error response
            return {
                "success": False,
                "data": [],
                "error": {
                    "code": e.code,
                    "message": e.message
                }
            }

In [3]:
#@title Create the connections as a dictionary

unique_values = {'northwind', 'openstreetmap', 'offshoreleaks', 'movies', 'companies', 'gameofthrones', 'network', 'recommendations', 'fincen', 'twitch', 'twitter', 'eoflix', 'grandstack', 'buzzoverflow', 'bluesky', 'stackoverflow2', 'stackoverflow'}
cred_to_connection = dict()
for unique_value in unique_values:
    cred_to_connection[unique_value] = Neo4jConnection(
        uri="neo4j+s://demo.neo4jlabs.com:7687",
        user=unique_value,
        password=unique_value,
        database=unique_value,
    )

In [4]:
#@title Test a connection with a query

cred_to_connection["fincen"].structured_query("MATCH (n) RETURN COUNT(n)")

{'success': True, 'data': [{'COUNT(n)': 7524}], 'error': None}

### Data Prep

In [6]:
import re
from datasets import load_dataset, Dataset
from deepdiff import DeepDiff
from llama_index.core.utils import get_tokenizer

# Load and prep dataset
SYSTEM_PROMPT = """
You will be asked a question by the user for which you are to generate a cypher query, keeping in mind the schema and the question provided.
You are to think, analyze and reason with the schema and question provided, only after which shall you generate the appropriate cypher query.
Follow the following format and wrap your reasoning and analysis inside the <reasoning> and </reasoning> tags and the final appropriate cypher
query inside <answer> and </answer>.
**LIMIT YOUR REASONING TO A MAX OF 400 WORDS/TOKENS**
Following the below format is imperative:
<reasoning>
...
</reasoning>
<answer>
...
</answer>
"""

cypher_prompt = """
Below is a database Neo4j schema and a question related to that database. Write a Cypher query to answer the question.

### Schema:
{schema}

### Question:
{question}

### Cypher:
"""

def extract_xml_answer(text: str) -> str:
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

prompt_mapping = dict()
# Dataset processing code
def process_example(example):
    formatted_pair = {
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': cypher_prompt.format(schema=example["schema"], question=example["question"])}
        ],
        'answer': example["cypher"]
    }
    alias = example["alias"]
    print('', end='   ')
    prompt_mapping[formatted_pair['prompt'][1]['content']] = alias
    return formatted_pair

# uncomment middle messages for 1-shot prompting
def get_data(split = "train") -> Dataset:
    dataset = load_dataset('neo4j/text2cypher-2024v1')[split]

    # Remove unnecessary columns
    columns_to_remove = ['data_source', 'instance_id']
    dataset = dataset.remove_columns(columns_to_remove)

    # Rename important column
    dataset = dataset.rename_column("database_reference_alias", "alias")

    #Remove rows with null values for `alias` column and process it
    dataset = dataset.filter(lambda example: example['alias'] is not None)
    dataset = dataset.map(lambda example: {'alias': example['alias'].split('_')[-1]})

    # Test for shortest 1000 rows for now
    dataset = dataset.sort(column_names = ["schema", "cypher", "question"])
    dataset = dataset.select(range(1000))

    # Finally, process the dataset for GRPO, also mapping each prompt to it's alias
    dataset = dataset.map(process_example)

    return dataset

dataset = get_data()

# Reward functions
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    responses = [completion[0]['content'] for completion in completions]
    q = prompts[0][-1]['content']
    extracted_responses = [extract_xml_answer(r) for r in responses]

    # Get alias, and subsequently the connection, for the question
    alias = prompt_mapping[q]
    connection = cred_to_connection[alias]

    # Get the response for the query from the connection
    answer = answer[0]
    correct_result = connection.structured_query(answer)["data"]

    # Also, process the queries in `extracted_responses`
    extracted_query_results = []
    for r in extracted_responses:
        query_result = connection.structured_query(r)
        extracted_query_results.append(query_result)

    # Compute similarity between results
    results = []
    for r in extracted_query_results:
      score = 0
      if not r["success"]:
        score = -0.5
      elif len(r["data"]) == 0:
        score = -0.25
      else:
        score = 1 - DeepDiff(correct_result, r["data"], ignore_order=True, get_deep_distance=True, cutoff_intersection_for_pairs=1)['deep_distance']
      results.append(score)

    return results

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward function that checks if the completion has a specific format."""
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, flags=re.DOTALL) for r in responses]
    return [0.5 if match else 0.0 for match in matches]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>\n") == 1:
        count += 0.125
    if text.count("\n</reasoning>\n") == 1:
        count += 0.125
    if text.count("\n<answer>\n") == 1:
        count += 0.125
        # count -= len(text.split("\n</answer>\n")[-1])*0.001
    if text.count("\n</answer>") == 1:
        count += 0.125
        # count -= (len(text.split("\n</answer>")[-1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

max_completion_length = 512
tokenizer_fn = get_tokenizer()
def completion_length_reward_func(completions, **kwargs):
    """
    Reward function that gives a reward of 1 when the completion length
    exactly matches max_completion_length, and decreases linearly to 0
    as the length deviates from max_completion_length.
    """
    rewards = []
    for completion in completions:
        content = completion[0]['content']
        length = len(tokenizer_fn(content))
        # Calculate the absolute difference ratio relative to max_completion_length
        diff_ratio = abs(max_completion_length - length) / max_completion_length
        # Reward is 1 if diff_ratio is 0, and linearly decreases toward 0 as diff_ratio increases.
        reward = max(0, 1 - diff_ratio)
        rewards.append(reward)
    return rewards

README.md:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/7.33M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/835k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/39554 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4833 [00:00<?, ? examples/s]

Filter:   0%|          | 0/39554 [00:00<?, ? examples/s]

Map:   0%|          | 0/22093 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
len(prompt_mapping.keys())

753

<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

In [8]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    use_vllm = True, # use vLLM for fast inference!
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    bf16 = is_bfloat16_supported(),
    fp16 = not is_bfloat16_supported(),
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 5, # Decrease if out of memory
    max_prompt_length = 4096,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 500,
    save_steps = 50,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 5


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [9]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        soft_format_reward_func,
        strict_format_reward_func,
        correctness_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 5 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (5 x 1 x 1) = 5
 "-____-"     Trainable parameters = 119,734,272/3,000,000,000 (3.99% trained)


Step,Training Loss,reward,reward_std,completion_length,kl,rewards / soft_format_reward_func,rewards / strict_format_reward_func,rewards / correctness_reward_func
1,19105.216800,0.625876,0.514785,273.000000,477630.468750,0.500000,0.000000,0.125876
2,0.011800,0.600000,0.876784,378.600006,0.295076,0.400000,0.000000,0.200000
3,0.006000,0.950000,0.758288,237.600006,0.149009,0.500000,0.000000,0.450000
4,0.005400,1.209303,0.170628,226.800003,0.135540,0.500000,0.000000,0.709303
5,0.005100,0.968872,0.821168,386.800018,0.127595,0.400000,0.000000,0.568873
6,0.005200,0.423077,0.387012,278.600006,0.129527,0.500000,0.000000,-0.076923
7,0.006500,0.539683,0.521024,250.800003,0.162867,0.500000,0.000000,0.039683
8,0.006100,-0.100000,0.223607,358.600006,0.151710,0.400000,0.000000,-0.500000
9,0.005000,1.061538,0.593522,212.199997,0.125781,0.500000,0.000000,0.561538
10,0.003400,0.950000,0.758288,180.600006,0.085060,0.500000,0.000000,0.450000


Unsloth: Will smartly offload gradients to save VRAM!


KeyboardInterrupt: 

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [14]:
example = dataset[-1]

text = tokenizer.apply_chat_template([
    {"role" : "user", "content" : cypher_prompt.format(schema=example["schema"], question=example["question"])},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 512,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

print(output)

Processed prompts: 100%|██████████| 1/1 [00:03<00:00,  3.46s/it, est. speed input: 161.69 toks/s, output: 11.26 toks/s]

```cypher
MATCH (u:User)-[r:WROTE]->(rev:Review)
ORDER BY r.stars DESC
LIMIT 1
RETURN u.name
```


In [13]:
example["question"]

'Which user wrote the review with the highest star rating?'

In [16]:
print(example["schema"])

Node properties:
- **Business**
  - `address`: STRING Available options: ['301 E Main St', '200 W Pine St', '313 N 1st St W', '1151 W Broadway St', '201 E Front St', '723 California Dr', '121 W Broadway St', '716 N San Mateo Dr', '111 E 5th Ave']
  - `location`: POINT 
  - `city`: STRING Available options: ['Missoula', 'Burlingame', 'San Mateo']
  - `state`: STRING Available options: ['MT', 'CA']
  - `businessId`: STRING Available options: ['b5', 'b3', 'b1', 'b2', 'b4', 'b7', 'b6', 'b8', 'b9']
  - `name`: STRING Available options: ['Missoula Public Library', "Ninja Mike's", 'KettleHouse Brewing Co.', 'Imagine Nation Brewing', 'Market on Front', 'Hanabi', 'Zootown Brew', "Ducky's Car Wash", 'Neo4j']
- **User**
  - `name`: STRING Available options: ['Will', 'Bob', 'Jenny', 'Angie']
  - `userId`: STRING Available options: ['u1', 'u2', 'u3', 'u4']
- **Review**
  - `date`: DATE Min: 2015-08-29, Max: 2018-09-10
  - `text`: STRING Available options: ['Not a great selection of books, but fortu

In [11]:
example['alias']

'grandstack'

In [12]:
example['cypher']

'MATCH (r:Review) WITH r ORDER BY r.stars DESC LIMIT 1 MATCH (u:User)-[:WROTE]->(r) RETURN u.name'

In [20]:
#@title Correct query result

cred_to_connection[example['alias']].structured_query("""
MATCH (r:Review) WITH r ORDER BY r.stars DESC LIMIT 1 MATCH (u:User)-[:WROTE]->(r) RETURN u.name
""")

{'success': True, 'data': [{'u.name': 'Jenny'}], 'error': None}

In [21]:
#@title SFT Trained query result

cred_to_connection[example['alias']].structured_query("""
MATCH (u:User)-[r:WROTE]->(rev:Review)
ORDER BY r.stars DESC
LIMIT 1
RETURN u.name
""")

{'success': True, 'data': [{'u.name': 'Will'}], 'error': None}

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [23]:
model.save_lora("grpo_saved_lora")

Now we load the LoRA and test:

In [28]:
text = tokenizer.apply_chat_template([
    {"role" : "system", "content" : SYSTEM_PROMPT},
    {"role" : "user", "content" : cypher_prompt.format(schema=example["schema"], question=example["question"])},
], tokenize = False, add_generation_prompt = True)

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 0.8,
    top_p = 0.95,
    max_tokens = 512,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

Processed prompts: 100%|██████████| 1/1 [00:08<00:00,  8.50s/it, est. speed input: 80.28 toks/s, output: 32.25 toks/s]


In [29]:
print(output)

<reasoning>
To answer this question, we need to find the user who wrote the review with the highest star rating. This requires accessing the relationships between users and reviews, as well as the properties of the reviews. The relationships `(:User)-[:WROTE]->(:Review)` will help us identify which user wrote which review, and the relationship `(:Review)-[:REVIEWS]->(:Business)` will allow us to access the review properties.
The `:Review` node properties `stars` will help us determine which review has the highest star rating.
The Cypher query will:
1. Match the relationships `(:User)-[:WROTE]->(:Review)` and `(:Review)-[:REVIEWS]->(:Business)`
2. Order the reviews by the `stars` property in descending order to find the highest-rated review
3. Limit the results to the first review to get the review with the highest star rating
4. Return the `name` property of the user who wrote this review
</reasoning>
<answer>
MATCH (u:User)-[:WROTE]->(r:Review)-[:REVIEWS]->(b:Business)
WITH r
ORDER BY

In [30]:
#@title GRPO tuned query result

cred_to_connection[example['alias']].structured_query("""
MATCH (u:User)-[:WROTE]->(r:Review)-[:REVIEWS]->(b:Business)
WITH r
ORDER BY r.stars DESC
LIMIT 1
MATCH (u:User)-[:WROTE]->(r)
RETURN u.name AS userName
""")

{'success': True, 'data': [{'userName': 'Jenny'}], 'error': None}

Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/drive/1WZDi7APtQ9VsvOrQSSC5DDtxq159j8iZ?usp=sharing)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp or a UI based system like Jan or Open WebUI. You can install Jan [here](https://github.com/janhq/jan) and Open WebUI [here](https://github.com/open-webui/open-webui)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
